## L2 regularization vs Weight Decay

Weight decay, also known as L2 regularization, is a fundamental technique used in deep learning to improve model performance. It acts as a regularizer that penalizes large weights in the network, leading to several benefits:

The weight decay penalty is typically implemented in one of two ways:

- L2 regularization: This directly adds a term to the loss function that is proportional to the sum of the squared weights.
- Weight decay in the optimizer: This modifies the update rule of the optimizer to include a decay factor that reduces the weights at each update step.

<div style="display:flex;">
<img src="./assets/SGD-weight_decay_equals_L2_reg.jpg" width="50%"/>
<img src="./assets/SGD_M_AdamW.jpg" width="50%"/>
</div>

When you see lerp_ inside the AdamW source code, it's being used as a mathematically elegant (and computationally fast) way to calculate the Running Exponential Moving Average of the gradients.

1. The Mathematical TranslationThe standard formula for the first moment 
($m_t$) in Adam is:$$m_t = \beta_1 m_{t-1} + (1 - \beta_1)g_t$$

If we rewrite this to match the lerp (Linear Interpolation) formula, where 

$\text{lerp}(start, end, weight) = start + weight \times (end - start)$:

1.Start: exp_avg (the old moving average)
2.End: grad (the new gradient)
3. Weight: 1 - beta1

Substituting these into the lerp formula:

- $\text{out} = exp\_avg + (1 - \beta_1) \times (grad - exp\_avg)$
- $\text{out} = exp\_avg + (1 - \beta_1)grad - exp\_avg + \beta_1 exp\_avg$
- $\text{out} = \beta_1 exp\_avg + (1 - \beta_1)grad$


It is mathematically identical.

2. Why use lerp_ instead of the standard formula?PyTorch developers use lerp_ (the in-place version) for two main reasons:
- Memory Efficiency: lerp_ performs the operation in-place. It doesn't create intermediate tensors for $(1 - \beta)$ or the result of the multiplication, which is crucial when training large models where every byte of VRAM counts.Performance 
- (Fusing): lerp is often implemented as a single "fused" kernel at the CUDA level. Instead of the GPU performing an addition, then a subtraction, then a multiplication, it does the whole interpolation in one pass over the data.

3. Visualizing the UpdateIn AdamW, these two lines handle the "memory" of the optimizer:

| Code Line | Purpose | What it represents |
|:-----|:------:|----:|
| exp_avg.lerp_(grad, 1 - beta1_t) | Momentum | Smooths out the direction of the gradient to avoid oscillations.
| exp_avg_sq.lerp_(grad.square(), 1 - beta2_t) | Volatility | Tracks how much the gradient is "vibrating" to scale the learning rate for each parameter.

In the optim.py file we are using 0-D CPU tensors to avoid recompilation of the compiled graph (need to understand more about torch.compile, dynamo and graph breaks)

https://docs.pytorch.org/docs/stable/user_guide/torch_compiler/compile/programming_model.recompilation.html#wrapping-constants-with-tensors

## Adam

<img src="./assets/Adam.png"/>

https://medium.com/@weidagang/data-compression-with-low-rank-approximation-using-neural-networks-d6a8e8426101

## AdamW was efficient and well proven, so why do we need a new optimizer?

- From the image above for every parameters of the neural network we have to keep track and calculate the momentum and the squared gradient. So AdamW takes twice as much memory as the model itself

- Adam treats all the parameters as a single long vector, updating each values independently without considering any internal structure - vector based optimizer

- To update the weights we first calculate the momentum for each parameter. For linear layers the momentum(naturally a 2D matrix) becomes almost low rank in practice. This means that only a small number of dominant directions really drive the updates - other directions contribute very little.


#### Thus, here comes the need for orthogonalization.

- Orthogonalization effectively increases the scale of other 'rare directions' which have small magnitude in the update but nevertheless they are important for learning. So we need to normalize the gradients too.


### Muon
#### The Key Idea - Orthogonalizing Updates

Muon takes the momentum update and orthogonalizes it before applying it to the weights.

- What is orthogonalization? - In linear algebra, an orthogonal matrix has rows (or columns) that are perpendicular and unit-length – like basis vectors in a coordinate system that don't interfere with each other. For a matrix U, it's orthogonal if U_T*U = I (identity matrix).

- Why orthogonalize? - Gradients and momentum updates in neural nets often have a high condition number - meaning they are stretched unevenly with most energy in a few directions(almost low-rank). This is like a squished ball that only rolls well in one way, ignoring other important paths. Muon fixes this by transforming the update to it's nearest semi-orthogonal version. Intuitively, it's like reshaping that squished ball into a perfect sphere, ensuring updates explore all directions evenly. The creators speculate this boosts "rare directions" that are crucial for learning but get drowned out otherwise.

- How it is done mathematically?  If the momentum update is G, Muon computes G = UΣVT (singular value decomposition, or SVD), then replaces it with U*V_transpose, essentially setting all singular values to 1, making it semi-orthogonal.

#### Newton-Schulz Iteration – The Efficient Orthogonalizer(odd polynomial matrix)

Direct SVD is computationally expensive, so Muon approximates orthogonalization with the Newton-Schulz (NS) iteration – a fast, iterative method.

- **How NS works intuitively**: Start with your update matrix G, normalize it (divide by its norm to scale singular values between 0 and 1), then repeatedly apply a polynomial update: X←aX + bX where B is derived from X*X_transpose and tuned coefficients a,b,c.
    
    Each step pushes the matrix closer to orthogonal. After 5 steps (as in Muon), it's a good approximation of U*V_transpose
    
- **Tuned coefficients**: The defaults are a=3.4445a , b=−4.7750b , c=2.0315.
These were optimized to converge quickly, especially for small singular values, while keeping the result close to unit scale.
- **Why NS over alternatives?**
    - SVD: Too slow for big matrices.
    - Other iterations (like coupled Newton): Unstable in low precision (e.g., bfloat16), which is key for GPU efficiency.
    
    NS runs stably in bfloat16, adding minimal overhead (under 1% FLOPs for typical setups like NanoGPT or large models like Llama 405B).

#### Key steps in Muon:
- The formulation of Muon is really simple; instead of keeping track of the first and second momentum, it only has the momentum(Nesterov momentum). It normalizes the momentum matrix before doing the weight update.

- Normalization can be done by Newton-Schulz or we can have a look at Polar Express.

In [1]:
def newton_schulz(G, steps=5, eps=1e-7):
  assert G.ndim == 2
  a, b, c = (3.445, -4.7750, 2.0315)
  X = G.bfloat(16)
  X /= (X.norm() + eps)

  if G.size(0) > G.size(1):
    X = X.T
  
  for _ in range(steps):
    A = X @ X.T
    B = b * A + c * A @ A
    X = a * X + B @ X
  
  if G.size(0) > G.size(1):
    X = X.T
    
  return X

In [2]:
def muon_update(grad, momentum, beta=0.95, ns_steps=5, nesterov=True):
  momentum.lerp_(grad, 1- beta)
  update = grad.lerp_(momentum, beta) if nesterov else momentum

  if update.ndim == 4: # for the case of conv filters [Muon is for 2D]
    update = update.view(len(update), -1)
  
  update = newton_schulz(update, steps=ns_steps)
  update *= max(1, grad.size(-2) / grad.size(-1)) ** 0.5
  return update

 - One neat trick was the flipping of the matrix so that we always work with a fat matrix (larger columns than rows) rather than a tall matrix (smaller columns). This reduces the number of dimensions in the matrix X @ X.T

 - In the blog post, the authors used Nesterov momentum instead of SGD momentum. In the former, we first go in the direction of momentum and then go in the direction of gradients. This produces more stable updates as compared to having unstable updates from the gradients of a batch.

 - Another improvement as compared to Adam is having only one buffer instead of 2. Adam had a first and second moment buffer, but Muon only has one buffer for momentum. That reduces the memory footprint by 50% (from 8 bytes to 4 bytes) per parameter.

### Nesterov momentum

#### What is Nesterov momentum?
Nesterov’s Momentum, also known as Nesterov Accelerated Gradient (NAG), is an enhanced version of the traditional momentum optimization technique. Nesterov’s Momentum is designed to accelerate the convergence of gradient descent by incorporating a look-ahead feature into the update rule.

#### How does Nesterov momentum work ?
Traditional momentum optimization helps to accelerate gradient descent by considering the past gradients in the update rule. It does so by adding a fraction of the previous update vector to the current gradient, effectively building up velocity over time and allowing the optimizer to move faster through saddle points or flat regions in the loss landscape.

Nesterov’s Momentum improves upon this by making a subtle yet powerful change to the update rule. Instead of calculating the gradient at the current position, **Nesterov’s Momentum calculates the gradient at a position slightly ahead in the direction of the accumulated momentum.** This look-ahead step allows the optimizer to correct its course more responsively if it is heading towards a suboptimal direction.

Traditional momentum involves maintaining an additional variable that represents the last update performed to the variable, an exponentially decaying moving average of past gradients.

This last update or last change to the variable is then added to the variable scaled by a “momentum” hyperparameter that controls how much of the last change to add, e.g. 0.9 for 90%.

It is easier to think about this update in terms of two steps, e.g calculate the change in the variable using the partial derivative then calculate the new value for the variable.

- change(t+1) = (momentum * change(t)) – (step_size * f'(x(t)))
- x(t+1) = x(t) + change(t+1)

A problem with momentum is that acceleration can sometimes cause the search to overshoot the minima at the bottom of a basin or valley floor.

**Nesterov Momentum can be thought of as a modification to momentum to overcome this problem of overshooting the minima.**

Nesterov Momentum is easy to think about this in terms of the four steps:

1. Project the position of the solution.
2. Calculate the gradient of the projection.
3. Calculate the change in the variable using the partial derivative.
4. Update the variable.

First, the projected position of the entire solution is calculated using the change calculated in the last iteration of the algorithm.
- projection(t+1) = x(t) + (momentum * change(t))

We can then calculate the gradient for this new position.
- gradient(t+1) = f'(projection(t+1))

Now we can calculate the new position of each variable using the gradient of the projection, first by calculating the change in each variable.
- change(t+1) = (momentum * change(t)) – (step_size * gradient(t+1))

And finally, calculating the new value for each variable using the calculated change.
- x(t+1) = x(t) + change(t+1)

In the field of convex optimization more generally, Nesterov Momentum is known to improve the rate of convergence of the optimization algorithm (e.g. reduce the number of iterations required to find the solution).